# Pivot and Unpivot

## Overview
**Pivoting** rotates rows into columns -- turning a long (tidy) table into a wide (crosstab) table. **Unpivoting** does the reverse.

**Long vs wide format:**
```
-- Long (tidy): one row per month-metric
month    metric       value
2023-01  deposits     2500
2023-01  withdrawals  800
2023-02  deposits     500

-- Wide (pivoted): one row per month, one column per metric
month    deposits  withdrawals  fees
2023-01  2500      800          0
2023-02  500       0            0
```

**Pivot approaches:**

| Approach | SQL | Notes |
|---|---|---|
| `CASE WHEN` | Standard SQL, works everywhere | Column values must be known in advance |
| `CROSSTAB` | PostgreSQL only (tablefunc extension) | Cleaner syntax, supports dynamic categories |
| Application-side | pandas `pivot_table()` | Most flexible for truly dynamic columns |

**PostgreSQL note:** `CREATE EXTENSION IF NOT EXISTS tablefunc` enables `CROSSTAB`. In SQLite, only the `CASE WHEN` approach is available. For unpivoting, PostgreSQL `CROSS JOIN LATERAL (VALUES ...)` is cleaner than `UNION ALL`.

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT, signup_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, opened_date TEXT, status TEXT, credit_limit REAL);CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, customer_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT);CREATE TABLE product_events (event_id INTEGER PRIMARY KEY, customer_id INTEGER, event_date TEXT, event_type TEXT, product TEXT, channel TEXT);INSERT INTO customers VALUES (1,'Aroha','Ngata','Retail','NB','2023-01-05',1),(2,'Liam','Chen','Wealth','NS','2023-01-12',1),(3,'Fatima','Al-Rashid','SME','ON','2023-01-20',1),(4,'James','MacLeod','Retail','NB','2023-02-03',1),(5,'Sofia','Petrov','Retail','BC','2023-02-14',1),(6,'Noah','Williams','SME','AB','2023-02-28',0),(7,'Mei','Zhang','Wealth','ON','2023-03-10',1),(8,'Ethan','Tremblay','Retail','QC','2023-04-05',1),(9,'Priya','Nair','Wealth','BC','2023-04-18',1),(10,'Marcus','Okafor','SME','ON','2023-04-25',1),(11,'Diana','Flores','Retail','NB','2023-05-02',1),(12,'Ravi','Patel','Retail','ON','2023-05-15',1),(13,'Yuki','Tanaka','Wealth','BC','2023-06-01',1),(14,'Omar','Hassan','SME','QC','2023-06-20',1),(15,'Chloe','Bouchard','Retail','QC','2023-07-08',0);INSERT INTO accounts VALUES (101,1,'Chequing','2023-01-05','Active',NULL),(102,1,'Savings','2023-01-05','Active',NULL),(103,2,'Investment','2023-01-12','Active',50000),(104,3,'Chequing','2023-01-20','Active',NULL),(105,3,'Loan','2023-01-20','Active',25000),(106,4,'Chequing','2023-02-03','Active',NULL),(107,5,'Chequing','2023-02-14','Active',NULL),(108,5,'Savings','2023-02-14','Active',NULL),(109,6,'Chequing','2023-02-28','Closed',NULL),(110,7,'Investment','2023-03-10','Active',75000),(111,8,'Chequing','2023-04-05','Active',NULL),(112,9,'Investment','2023-04-18','Active',100000),(113,10,'Chequing','2023-04-25','Active',NULL),(114,10,'Loan','2023-04-25','Active',15000),(115,11,'Chequing','2023-05-02','Active',NULL),(116,12,'Chequing','2023-05-15','Active',NULL),(117,12,'Savings','2023-05-15','Active',NULL),(118,13,'Investment','2023-06-01','Active',60000),(119,14,'Chequing','2023-06-20','Active',NULL),(120,15,'Chequing','2023-07-08','Closed',NULL);INSERT INTO transactions VALUES (1001,101,1,'2023-01-10','Deposit',2500.00,'Payroll'),(1002,101,1,'2023-01-22','Withdrawal',-800.00,'Rent'),(1003,102,1,'2023-02-05','Deposit',500.00,'Transfer'),(1004,103,2,'2023-02-10','Deposit',10000.00,'Investment'),(1005,104,3,'2023-02-15','Deposit',3200.00,'Payroll'),(1006,104,3,'2023-03-01','Withdrawal',-1200.00,'Rent'),(1007,106,4,'2023-03-10','Deposit',2800.00,'Payroll'),(1008,107,5,'2023-03-15','Deposit',2200.00,'Payroll'),(1009,107,5,'2023-03-28','Fee',-25.00,'Monthly Fee'),(1010,101,1,'2023-03-10','Deposit',2500.00,'Payroll'),(1011,103,2,'2023-04-15','Deposit',15000.00,'Investment'),(1012,110,7,'2023-04-20','Deposit',20000.00,'Investment'),(1013,111,8,'2023-05-01','Deposit',2100.00,'Payroll'),(1014,112,9,'2023-05-10','Deposit',25000.00,'Investment'),(1015,113,10,'2023-05-20','Deposit',3500.00,'Payroll'),(1016,115,11,'2023-06-01','Deposit',2000.00,'Payroll'),(1017,116,12,'2023-06-15','Deposit',2700.00,'Payroll'),(1018,118,13,'2023-07-01','Deposit',18000.00,'Investment'),(1019,101,1,'2023-07-10','Deposit',2500.00,'Payroll'),(1020,103,2,'2023-07-20','Deposit',12000.00,'Investment'),(1021,104,3,'2023-07-15','Deposit',3200.00,'Payroll'),(1022,107,5,'2023-08-01','Deposit',2200.00,'Payroll'),(1023,111,8,'2023-08-05','Deposit',2100.00,'Payroll'),(1024,113,10,'2023-08-15','Withdrawal',-500.00,'Transfer'),(1025,116,12,'2023-08-20','Deposit',2700.00,'Payroll'),(1026,101,1,'2023-10-10','Deposit',2500.00,'Payroll'),(1027,116,12,'2023-10-20','Deposit',2700.00,'Payroll');INSERT INTO product_events VALUES (1,1,'2023-01-10','page_view','Savings Account','web'),(2,1,'2023-01-10','product_view','Savings Account','web'),(3,1,'2023-01-10','application_start','Savings Account','web'),(4,1,'2023-01-11','application_submit','Savings Account','web'),(5,1,'2023-01-15','account_opened','Savings Account','web'),(6,2,'2023-01-12','page_view','Investment Account','mobile'),(7,2,'2023-01-12','product_view','Investment Account','mobile'),(8,2,'2023-01-12','application_start','Investment Account','mobile'),(9,2,'2023-01-13','application_submit','Investment Account','mobile'),(10,2,'2023-01-15','account_opened','Investment Account','mobile'),(11,3,'2023-01-20','page_view','Chequing Account','web'),(12,3,'2023-01-20','product_view','Chequing Account','web'),(13,3,'2023-01-20','application_start','Chequing Account','web'),(14,4,'2023-02-03','page_view','Chequing Account','web'),(15,4,'2023-02-03','product_view','Chequing Account','web'),(16,5,'2023-02-14','page_view','Savings Account','mobile'),(17,5,'2023-02-14','product_view','Savings Account','mobile'),(18,5,'2023-02-14','application_start','Savings Account','mobile'),(19,5,'2023-02-15','application_submit','Savings Account','mobile'),(20,5,'2023-02-20','account_opened','Savings Account','mobile'),(21,7,'2023-03-10','page_view','Investment Account','web'),(22,7,'2023-03-10','product_view','Investment Account','web'),(23,7,'2023-03-10','application_start','Investment Account','web'),(24,7,'2023-03-11','application_submit','Investment Account','web'),(25,7,'2023-03-15','account_opened','Investment Account','web'),(26,8,'2023-04-05','page_view','Chequing Account','mobile'),(27,8,'2023-04-05','product_view','Chequing Account','mobile'),(28,9,'2023-04-18','page_view','Investment Account','web'),(29,9,'2023-04-18','product_view','Investment Account','web'),(30,9,'2023-04-18','application_start','Investment Account','web'),(31,9,'2023-04-20','application_submit','Investment Account','web'),(32,9,'2023-04-25','account_opened','Investment Account','web');""")
conn.commit()
print("Finance schema ready.")
for t in ["customers","accounts","transactions","product_events"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

Finance schema ready.
  customers: 15 rows
  accounts: 20 rows
  transactions: 27 rows
  product_events: 32 rows


---
## CASE-based pivot: transaction counts by type and month

In [2]:
import pandas as pd
q = """
SELECT  STRFTIME('%Y-%m', txn_date)  AS month,
        COUNT(CASE WHEN txn_type = 'Deposit'    THEN 1 END)  AS deposits,
        COUNT(CASE WHEN txn_type = 'Withdrawal' THEN 1 END)  AS withdrawals,
        COUNT(CASE WHEN txn_type = 'Fee'        THEN 1 END)  AS fees,
        COUNT(*)                                              AS total
FROM    transactions
GROUP BY STRFTIME('%Y-%m', txn_date)
ORDER BY month
"""
print("=== Pivot: transaction count by type x month ===")
print(pd.read_sql(q, conn).to_string(index=False))

=== Pivot: transaction count by type x month ===
  month  deposits  withdrawals  fees  total
2023-01         1            1     0      2
2023-02         3            0     0      3
2023-03         3            1     1      5
2023-04         2            0     0      2
2023-05         3            0     0      3
2023-06         2            0     0      2
2023-07         4            0     0      4
2023-08         3            1     0      4
2023-10         2            0     0      2


---
## Pivot with SUM: deposits by segment and month

In [3]:
import pandas as pd
q = """
SELECT  STRFTIME('%Y-%m', t.txn_date)  AS month,
        ROUND(SUM(CASE WHEN c.segment='Retail' AND t.txn_type='Deposit'
                       THEN t.amount ELSE 0 END), 2)  AS retail_deposits,
        ROUND(SUM(CASE WHEN c.segment='Wealth' AND t.txn_type='Deposit'
                       THEN t.amount ELSE 0 END), 2)  AS wealth_deposits,
        ROUND(SUM(CASE WHEN c.segment='SME'    AND t.txn_type='Deposit'
                       THEN t.amount ELSE 0 END), 2)  AS sme_deposits,
        ROUND(SUM(CASE WHEN t.txn_type='Deposit' THEN t.amount ELSE 0 END), 2) AS total_deposits
FROM    transactions AS t
JOIN    customers    AS c  ON t.customer_id = c.customer_id
WHERE   t.txn_type = 'Deposit'
GROUP BY STRFTIME('%Y-%m', t.txn_date)
ORDER BY month
"""
print("=== Pivot: total deposits by segment x month ===")
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("PostgreSQL CROSSTAB equivalent (requires tablefunc extension):")
print("""
  CREATE EXTENSION IF NOT EXISTS tablefunc;

  SELECT * FROM CROSSTAB(
    $$SELECT STRFTIME('%Y-%m', t.txn_date), c.segment, ROUND(SUM(t.amount),2)
      FROM transactions t JOIN customers c USING (customer_id)
      WHERE t.txn_type = 'Deposit'
      GROUP BY 1, 2 ORDER BY 1, 2$$,
    $$SELECT DISTINCT segment FROM customers ORDER BY 1$$
  ) AS ct (month TEXT, "Retail" NUMERIC, "SME" NUMERIC, "Wealth" NUMERIC);
""")

=== Pivot: total deposits by segment x month ===
  month  retail_deposits  wealth_deposits  sme_deposits  total_deposits
2023-01           2500.0              0.0           0.0          2500.0
2023-02            500.0          10000.0        3200.0         13700.0
2023-03           7500.0              0.0           0.0          7500.0
2023-04              0.0          35000.0           0.0         35000.0
2023-05           2100.0          25000.0        3500.0         30600.0
2023-06           4700.0              0.0           0.0          4700.0
2023-07           2500.0          30000.0        3200.0         35700.0
2023-08           7000.0              0.0           0.0          7000.0
2023-10           5200.0              0.0           0.0          5200.0

PostgreSQL CROSSTAB equivalent (requires tablefunc extension):

  CREATE EXTENSION IF NOT EXISTS tablefunc;

  SELECT * FROM CROSSTAB(
    $$SELECT STRFTIME('%Y-%m', t.txn_date), c.segment, ROUND(SUM(t.amount),2)
      FROM transa

---
## Unpivot: wide to long with UNION ALL

In [4]:
import pandas as pd
# Build wide summary table to unpivot
conn.execute("""
CREATE TEMP TABLE monthly_summary AS
SELECT STRFTIME('%Y-%m', txn_date) AS month,
       ROUND(SUM(CASE WHEN txn_type='Deposit'    THEN amount ELSE 0 END),2) AS deposits,
       ROUND(SUM(CASE WHEN txn_type='Withdrawal' THEN amount ELSE 0 END),2) AS withdrawals,
       ROUND(SUM(CASE WHEN txn_type='Fee'        THEN amount ELSE 0 END),2) AS fees
FROM   transactions GROUP BY 1 ORDER BY 1""")
conn.commit()
print("Wide format:")
print(pd.read_sql("SELECT * FROM monthly_summary", conn).to_string(index=False))
q = """
SELECT month, 'deposits'    AS metric, deposits    AS value FROM monthly_summary
UNION ALL
SELECT month, 'withdrawals' AS metric, withdrawals AS value FROM monthly_summary
UNION ALL
SELECT month, 'fees'        AS metric, fees        AS value FROM monthly_summary
ORDER BY month, metric
"""
print()
print("Long (unpivoted) format:")
print(pd.read_sql(q, conn).to_string(index=False))
print()
print("PostgreSQL VALUES-based unpivot (cleaner than UNION ALL for many columns):")
print("""
  SELECT month, metric, value
  FROM   monthly_summary
  CROSS JOIN LATERAL (VALUES
      ('deposits',    deposits),
      ('withdrawals', withdrawals),
      ('fees',        fees)
  ) AS v(metric, value)
  ORDER BY month, metric;
""")
conn.close()

Wide format:
  month  deposits  withdrawals  fees
2023-01    2500.0       -800.0   0.0
2023-02   13700.0          0.0   0.0
2023-03    7500.0      -1200.0 -25.0
2023-04   35000.0          0.0   0.0
2023-05   30600.0          0.0   0.0
2023-06    4700.0          0.0   0.0
2023-07   35700.0          0.0   0.0
2023-08    7000.0       -500.0   0.0
2023-10    5200.0          0.0   0.0

Long (unpivoted) format:
  month      metric   value
2023-01    deposits  2500.0
2023-01        fees     0.0
2023-01 withdrawals  -800.0
2023-02    deposits 13700.0
2023-02        fees     0.0
2023-02 withdrawals     0.0
2023-03    deposits  7500.0
2023-03        fees   -25.0
2023-03 withdrawals -1200.0
2023-04    deposits 35000.0
2023-04        fees     0.0
2023-04 withdrawals     0.0
2023-05    deposits 30600.0
2023-05        fees     0.0
2023-05 withdrawals     0.0
2023-06    deposits  4700.0
2023-06        fees     0.0
2023-06 withdrawals     0.0
2023-07    deposits 35700.0
2023-07        fees     0.0
202

---
## Common Pitfalls

**1. Hard-coding pivot column values that change over time**
`CASE WHEN month = '2023-01' THEN ...` breaks when January 2023 is no longer in the data or a new month appears. For dynamic pivots where column values are unknown in advance, generate the SQL programmatically (Python f-string, dbt macro) or use PostgreSQL `CROSSTAB`.

**2. Using SUM instead of COUNT producing NULLs instead of zeros**
`SUM(CASE WHEN txn_type = 'Fee' THEN 1 END)` returns NULL when there are no fees -- because SUM of an empty set is NULL. Use `COALESCE(SUM(...), 0)` or `COUNT(CASE WHEN txn_type = 'Fee' THEN 1 END)`, which returns 0 for no matching rows.

**3. Join fan-out before aggregation inflating pivot values**
Joining transactions to customers before the GROUP BY can multiply rows if the join is not strictly many-to-one. Aggregate first in a CTE, then enrich with labels -- or verify the join cardinality before pivoting.

**4. UNION ALL column count and type mismatch in unpivot**
Every branch of the UNION ALL must have the same number of columns with compatible types. Mixing REAL in one branch and TEXT in another causes a type error in PostgreSQL or silent coercion in SQLite. Cast explicitly: `CAST(deposits AS TEXT)`.

**5. PostgreSQL CROSSTAB requires input sorted by row key then category**
`CROSSTAB` requires its inner query sorted `ORDER BY row_identifier, category_value`. Wrong sort order assigns values to the wrong columns -- silent data corruption with no error. Always test CROSSTAB output against a manually verified subset.

**6. Losing NULL vs zero distinction during unpivot**
`CASE WHEN txn_type='Fee' THEN amount ELSE 0 END` turns "no fees this month" into 0 rather than NULL. This matters for averages: `AVG` skips NULLs but includes zeros. Use NULL when the value is genuinely absent; use 0 only when you mean zero activity.


---
*sql_methods_library - Samantha McGarrigle*